# 01 Private and Public

A privacy-sensitive project splits every data and output folder into `private/` and
`public/`. Private folders never reach git; public folders hold only reviewed,
aggregated results. This notebook treats the study's participant records as
identifiable and walks them from a private folder to a public summary.

In [ ]:
import shutil
import subprocess

import framework as fw
import matplotlib.pyplot as plt

root = fw.scaffold()

## The layout

`security.sensitive` is on, and each folder has a private and a public twin.

In [ ]:
print("sensitive:", fw.settings("security.sensitive"))
info = fw.project_info()
info[info["key"].str.contains("private|public")]

## What git ignores

`git check-ignore` shows which rule covers a path. Private folders, `.env`, the cache,
and `framework.db` are ignored; public folders are not.

In [ ]:
paths = [
    "inputs/private/raw/participants.csv",
    "outputs/private/tables/bp_by_participant.csv",
    "outputs/private/cache/model.pickle",
    "inputs/public/final/bp_by_site.csv",
    "outputs/public/tables/bp_by_site.csv",
    ".env",
    "framework.db",
]
for path in paths:
    rule = subprocess.run(["git", "check-ignore", "-v", path], cwd=root,
                          capture_output=True, text=True).stdout.strip()
    print(f"{path:48} {'ignored by ' + rule.split(chr(9))[0] if rule else 'tracked'}")

There is no built-in encryption: protection comes from keeping files in private folders,
out of git, and out of published work.

## Private inputs

Record-level data goes into `inputs/private/raw/`. `data_add()` names each file from
its folder (`inputs.private.raw.participants`), and `data_info()` marks it private.

In [ ]:
shared_csv = root.parent.parent / "shared" / "csv"
private_raw = root / fw.settings("inputs_private_raw")
for name in ("participants", "visits", "sites"):
    shutil.copy(shared_csv / f"{name}.csv", private_raw / f"{name}.csv")
    fw.data_add(private_raw / f"{name}.csv")

fw.data_info("inputs.private.raw.participants")

`data_info(columns=True)` shows a private file's layout (column types and missing
counts) without any values.

In [ ]:
fw.data_info("inputs.private.raw.participants", columns=True).columns

Work with the records without printing them. Shapes and counts are fine to show.

In [ ]:
participants = fw.data_read("inputs.private.raw.participants")
visits = fw.data_read("inputs.private.raw.visits")
sites = fw.data_read("inputs.private.raw.sites")
records = visits.merge(participants, on="participant_id").merge(sites, on="site_id")
records.shape

The joined records are saved to the private intermediate folder. The catalog name's
leading parts pick the folder.

In [ ]:
fw.data_save(records, "inputs.private.intermediate.bp_records");
fw.data_info("inputs.private.intermediate.bp_records").path

## A public summary

Aggregate to site, arm, and visit, and suppress small cells before anything goes
public. The threshold is 15 participants here, high enough to show suppression on this
small cohort.

In [ ]:
summary = (
    records.dropna(subset=["systolic_bp"])
    .groupby(["site_name", "treatment", "visit_number"])
    .agg(n=("participant_id", "nunique"), mean_systolic=("systolic_bp", "mean"))
    .round(1)
    .reset_index()
)
small = summary["n"] < 15
summary.loc[small, ["n", "mean_systolic"]] = None
print(f"{small.sum()} of {len(summary)} cells suppressed")
fw.data_save(summary, "inputs.public.final.bp_by_site", type="csv");
summary.head()

## Private and public outputs

`save_table()` and the other `save_*()` functions write to the private folder unless
`public=True`. Look at the two paths. This session has read private data, so each
`public=True` save warns, as a reminder to check that what you save is an aggregate.

In [ ]:
print(fw.save_table(records.head(50), "bp_records_sample"))
print(fw.save_table(summary, "bp_by_site", public=True))

In [ ]:
means = summary.groupby(["visit_number", "treatment"])["mean_systolic"].mean().unstack()
fig, ax = plt.subplots(figsize=(6, 4))
means.plot(ax=ax, marker="o", xlabel="Visit", ylabel="Mean systolic BP (mmHg)")
fw.save_figure(fig, "bp_by_visit", public=True, width=6, height=4)

`result_list()` records which results are public.

In [ ]:
fw.result_list()[["name", "type", "public"]]

## Publishing

Publishing checks the path first. A file whose path contains "private" stops in a
notebook (at a terminal it asks) unless you pass `include_private=True`.

In [ ]:
try:
    fw.publish(root / "outputs/private/tables/bp_records_sample.csv",
               slug="should-not-publish")
except fw.FrameworkError as error:
    print(error)

In a sensitive project, every object and every notebook also needs confirmation, even
from a public path, because it could show private data. In a notebook these stop.

In [ ]:
try:
    fw.publish(summary, slug="bp-by-site-table")
except fw.FrameworkError as error:
    print(error)

In [ ]:
try:
    fw.publish_notebook(root / "notebooks/01-private-and-public.ipynb")
except fw.FrameworkError as error:
    print(error)

Three reasons come back for the notebook: the project is sensitive, its code looks like
it reads private data, and its own file name contains "private" (the path check
matching the name, not a private folder).

After review, `include_private=True` is the confirmation. The suppressed site summary is
safe, so it is published to the `local` folder destination (`published/`).

In [ ]:
fw.publish(summary, slug="bp-by-site-table", title="Systolic BP by site (suppressed)",
           include_private=True)

A file in a public folder that isn't a notebook publishes without a prompt.

In [ ]:
fw.publish(root / "outputs/public/figures/bp_by_visit.png",
           title="Systolic BP by visit (figure)")

## Checks

Sensitive projects turn on the `data_security` and `check_sensitive_dirs` hook checks in
`settings/git.yml`. The hook is installed in a project that is its own git repository;
this one sits inside the shared test repository, so the checks show as enabled but not
installed, and installing refuses.

In [ ]:
fw.git_hooks_list()

In [ ]:
try:
    fw.git_hooks_install()
except fw.FrameworkError as error:
    print(str(error).splitlines()[0])

`git_security_audit()` confirms private folders are ignored, no private data is tracked,
and no tracked data file has identifier-like columns.

In [ ]:
fw.git_security_audit(history_depth="shallow", verbose=False).summary